# Dynamic Connectedness and Hedging Role of Gold under Geopolitical Risk

本笔记本是**基于 R 的论文级实证框架**：前半部分用 Markdown 交代**经济含义、识别思路与写作结构**；后半部分用 **Coding** 串联可复现流程。

- **数据**：开头统一用**代称对象**（核心面板为 `R_assets` / `GPR_daily`，列名见 `ASSET_LABELS`）与**占位样本**；你只需在标记为 `<<< 用户替换 >>>` 的代码块中导入真实价格/指数并计算对数收益即可。
- **模型**：TVP–VAR–connectedness、DCC–GARCH 等在此处给出**接口与占位**；占位返回 `NA` 或简化统计量，避免在无专用包时强行估计导致报错。
- **原则**：全样本推断仅使用**当期及以前信息**（滚动窗口、递归样本），以规避 **look-ahead bias**。


## 0. 实证路线图（与论文章节对应）

| 步骤 | 内容 | 本 notebook |
|------|------|-------------|
| A | 日频五类资产对数收益 + 月频 GPR 对齐 | 代称 + 占位生成；预留替换 |
| B | 描述统计（mean / sd / skew / kurt / JB / ADF） | 可运行 |
| C | 动态连通性（TVP–VAR–DY）：TCI、网络、黄金 NET 动态 | 理论说明 + 函数壳 + 静态相关作图 |
| D | GPR 高低状态子样本 | 分位阈值 `q_GPR_high` |
| E | 组合再平衡：1N、MVP（滚动协方差）、MCP / MCoP 占位 | 1N + 滚动 MVP 可跑；DCC / 连通性权重为占位 |
| F | 指标：均值、波动、对冲有效性、夏普 | 可跑（无风险利率可设 0） |

**记号**：下文用 **A1–A5** 表示五类资产列顺序：`GOLD`, `OIL_WTI`, `USD_IDX`, `SP500`, `UST_TSY`。


## 1. 引言（Markdown）：问题与贡献

**研究问题**：在地缘政治风险（GPR）高低状态下，黄金与其他大类资产之间的**动态连通性**如何演变？黄金是否承担**系统层面的风险吸收或溢出枢纽**角色（`NET`、`FROM`、`TO`）？进一步，在**日频再平衡**规则下，纳入黄金是否改善**对冲有效性与风险调整收益**？

**识别要点**：（1）用 **TVP–VAR–connectedness** 刻画时变总溢出与方向性溢出；（2）用 **GPR > q0.7** 划分 `high` / `normal`，比较子样本；（3）用多种**权重规则**（等权、滚动最小方差、基于相关矩阵与成对连通性的规则）构造组合，并在 **with / without gold** 下比较 **夏普、波动与对冲有效性**。

以下各节在 **Coding** 中给出与上表一致的实现骨架。


In [ ]:

## ---------------------------------------------------------------------------
## 2. 环境与参数（Coding）
## ---------------------------------------------------------------------------
options(stringsAsFactors = FALSE)
options(repos = c(CRAN = "https://cloud.r-project.org"))

suppressPackageStartupMessages({
  library(xts)
  library(zoo)
  library(PerformanceAnalytics)
})

ensure_pkg <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
  invisible(TRUE)
}

ensure_pkg("tseries")

## 全局参数（与论文设定一致；可按需修改）
SAMPLE_START <- as.Date("2000-01-01")
SAMPLE_END   <- Sys.Date()   ## latest available date（占位：运行日）
WINDOW_MVP   <- 252L         ## 滚动协方差窗口（交易日近似）
REBAL_FREQ   <- 1L           ## 日频再平衡：每 1 日调仓
Q_GPR_HIGH   <- 0.70         ## GPR 高分位：GPR > q0.7 -> high
RF_DAILY     <- 0 / 252      ## 无风险利率（日化，占位：可改为真实短端利率序列）

ASSET_LABELS <- c(
  "GOLD",      ## 黄金
  "OIL_WTI",   ## 原油 WTI
  "USD_IDX",   ## 美元指数
  "SP500",     ## 标普 500
  "UST_TSY"    ## 美国国债指数（Treasury index）
)

message("参数就绪：样本 [", SAMPLE_START, ", ", SAMPLE_END, "]；资产列：", paste(ASSET_LABELS, collapse = ", "))


## 3. 数据：代称、处理规则与占位（Markdown + Coding）

**日频五类资产**：对价格水平 $P_t$ 计算对数收益 $r_t = \ln(P_t) - \ln(P_{t-1})$。

**月频 GPR**：构造日频对齐序列时，采用**当月 GPR 值填充该月所有交易日**（或你论文中采用的其它规则），并确保在时点 $t$ 做决策时**不使用 $t$ 之后**才公布的 GPR 修订值。

**本节 Coding**：不读入外部文件；用**可重复的占位随机游走收益**生成 `returns_daily_xts` 与 `gpr_monthly_xts`，对象名与列名与正式实证一致。你可在下一单元整体替换为 `read.csv` / `quantmod::getSymbols` 等。


In [ ]:

## ---------------------------------------------------------------------------
## 3.1 占位数据生成（<<< 用户替换 >>>）
## ---------------------------------------------------------------------------
set.seed(20260419)

## --- 交易日日历（简化：逐日序列；你可改为实际交易日索引） ---
all_days <- seq(from = SAMPLE_START, to = SAMPLE_END, by = "day")
## 粗略去掉周末（正式实证请换 business day 日历）
is_bd <- weekdays(all_days, abbreviate = TRUE) %in% c("Mon", "Tue", "Wed", "Thu", "Fri")
trade_dates <- all_days[is_bd]
n <- length(trade_dates)
if (n < WINDOW_MVP + 50) {
  ## 若样本太短（例如 SAMPLE_END 很近），向前扩展占位长度以便演示滚动窗口
  trade_dates <- seq.Date(from = SAMPLE_START, by = "day", length.out = WINDOW_MVP + 800)
  is_bd <- weekdays(trade_dates, abbreviate = TRUE) %in% c("Mon", "Tue", "Wed", "Thu", "Fri")
  trade_dates <- trade_dates[is_bd]
  n <- length(trade_dates)
}

## 代称：五类资产日对数收益矩阵（占位）
R_noise <- matrix(rnorm(n * length(ASSET_LABELS), sd = 0.008), nrow = n, ncol = length(ASSET_LABELS))
colnames(R_noise) <- ASSET_LABELS
returns_daily_xts <- xts(R_noise, order.by = trade_dates)

## 代称：月频 GPR（占位；单调随机游走 + 噪声）
m_end <- seq(from = as.Date(format(min(trade_dates), "%Y-%m-01")), length.out = ceiling(n / 20) + 3, by = "month") - 1
m_end <- m_end[m_end <= max(trade_dates)]
gpr_raw <- exp(cumsum(rnorm(length(m_end), sd = 0.04)))
gpr_monthly_xts <- xts(gpr_raw, order.by = m_end)

## 对齐到日频：当月值填充（na.locf 自月初；此处用月末索引近似）
gpr_daily <- na.locf(merge.xts(returns_daily_xts[, 1], gpr_monthly_xts))[, 2, drop = FALSE]
colnames(gpr_daily) <- "GPR"

data_panel_xts <- merge.xts(returns_daily_xts, gpr_daily)
data_panel_xts <- na.omit(data_panel_xts)

## 高低 GPR 指示（全样本分位，占位；正式实证需说明是否递归分位以避免前视）
q_cut <- quantile(as.numeric(data_panel_xts$GPR), probs = Q_GPR_HIGH, na.rm = TRUE)
regime_high_gpr <- data_panel_xts$GPR > q_cut
regime_low_gpr  <- data_panel_xts$GPR <= q_cut

## 对外暴露：与论文章节一致的代称（后续单元均使用这些符号）
R_assets <- data_panel_xts[, ASSET_LABELS]
GPR_daily <- data_panel_xts[, "GPR"]

message("占位面板维度：", nrow(data_panel_xts), " x ", ncol(data_panel_xts))


## 4. 数据分布与序列特征（Markdown）

**分析文字（写作时可扩写）**：五类资产收益的**尖峰厚尾**与**波动聚集**会影响 JB 与 ARCH 类检验；美元指数与股指之间常呈现**时变相关**。黄金在风险事件期的**左尾联动**是否增强，需结合 GPR **high** 子样本的**分位数图与滚动相关**讨论。

**Coding**：对占位数据画**时间序列**与**直方图**（正式写作请换主题与配色）。


In [ ]:

par(mfrow = c(2, 1), mar = c(3, 3, 2, 1), mgp = c(2, 0.7, 0))
plot(R_assets[, "GOLD"], main = "代称：GOLD 日对数收益（占位）", col = "#b8860b", lwd = 1)
plot(GPR_daily, main = "代称：GPR（对齐到日频，占位）", col = "#2e6f9e", lwd = 1)
par(mfrow = c(1, 1))

hist(as.numeric(R_assets[, "GOLD"]), breaks = 80, prob = TRUE,
     main = "GOLD 收益分布（占位）", xlab = "r", col = grDevices::gray(0.85))


## 5. 描述统计：mean / sd / skew / kurt / JB / ADF（Markdown）

**说明**：JB 与 ADF 调用 `tseries`；ADF 的原假设为**单位根**（对收益序列通常期望拒绝）。若你检验的是**价格水平**而非收益，解释需反转。

**Coding**：生成 `stats_desc` 表（`xts` 各列）。


In [ ]:

desc_one <- function(x) {
  x <- as.numeric(na.omit(x))
  n <- length(x)
  m <- mean(x)
  s <- sd(x)
  sk <- PerformanceAnalytics::skewness(x, method = "sample")
  kt <- PerformanceAnalytics::kurtosis(x, method = "sample")
  jb <- tseries::jarque.bera.test(x)
  ad <- tseries::adf.test(x, alternative = "stationary", k = trunc((length(x) - 1)^(1/3)))
  c(
    N = n, Mean = m, SD = s, Skew = sk, Kurt = kt,
    JB_stat = unname(jb$statistic), JB_p = jb$p.value,
    ADF_stat = unname(ad$statistic), ADF_p = ad$p.value
  )
}

stats_desc <- as.data.frame(t(apply(R_assets, 2, desc_one)))
print(round(stats_desc, 6))


## 6. 动态连通性：TVP–VAR–Diebold–Yilmaz（Markdown）

**模型叙述（正文可展开）**：在 $N$ 元（此处 $N=5$）结构 VAR 上，通过广义预测误差方差分解得到**方向性溢出**与**总连通性指数（TCI）**；时变参数（TVP）允许系数随时间漂移，从而得到**动态 TCI** 与**黄金净溢出（NET）**路径。

**本 notebook 的 Coding 策略**：
- 给出 `estimate_connectedness_tvp_var()` **占位函数**：返回 `NULL` 并在控制台提示替换为你的 TVP–VAR–DY 程序（例如自写 Gibbs 或现成 R 包流程）。
- 同时计算**全样本相关矩阵**并作 **heatmap**，作为“数据结构初检”图（**不能**替代正式溢出分解）。

**子样本**：`idx_full`、`idx_high`、`idx_low` 供后续表格与作图复用。


In [ ]:

estimate_connectedness_tvp_var <- function(R, lags = 2L, H = 10L) {
  message("[占位] TVP-VAR-connectedness：请替换为论文所用估计程序（lags、预测步长 H、Bayesian 先验等）。")
  message("      期望输出对象建议包含：spillover_table, TCI_t, NET_each_asset_t, FROM_TO_t。")
  invisible(NULL)
}

estimate_connectedness_tvp_var(R_assets)

C_full <- cor(as.matrix(na.omit(R_assets)))
cat("全样本相关矩阵（占位，非 DY 溢出）：
")
print(round(C_full, 3))

idx_full <- rep(TRUE, nrow(R_assets))
idx_high <- as.logical(regime_high_gpr[index(R_assets)])
idx_low  <- as.logical(regime_low_gpr[index(R_assets)])

image(
  1:ncol(C_full), 1:nrow(C_full), t(C_full[nrow(C_full):1, ]),
  col = hcl.colors(12, "Blues 3"),
  xlab = "", ylab = "", axes = FALSE,
  main = "相关矩阵 heatmap（占位，非正式 spillover table）"
)
axis(1, at = 1:ncol(C_full), labels = colnames(C_full), las = 2)
axis(2, at = 1:nrow(C_full), labels = rev(colnames(C_full)), las = 2)


## 7. 组合优化与日频再平衡（Markdown）

**策略**：
- **1N**：等权。
- **MVP**：滚动样本协方差 $\hat{\Sigma}_{t}$ 仅在 $\{t-W+1,\ldots,t\}$ 上估计，在 $t+1$ 使用 $\hat{w}_{t+1}\propto \hat{\Sigma}_t^{-1}\mathbf{1}$（实现中对正定矩阵求逆，奇异时退回等权）。
- **MCP / MCoP / DCC–GARCH**：此处**仅占位**——分别对应“基于相关矩阵的规则权重”和“基于成对连通性矩阵的规则权重”以及“DCC 预测协方差”；你接入专用包后把权重函数填入 `weight_mcp()` 等。

**指标**：日均收益、年化波动（$\sqrt{252}\cdot\text{sd}$）、对冲有效性（相对基准方差下降比例，此处基准取 **1N**）、夏普（超额收益 / 波动）。

**子样本**：对 `idx_high`、`idx_low` 分别切片评估；**with / without gold** 通过列子集实现。


In [ ]:

## 滚动 MVP（无 DCC；真正 DCC 请 <<< 用户替换 >>>）
rolling_mvp_weights <- function(R, win = WINDOW_MVP) {
  R <- as.xts(R)
  n <- nrow(R)
  p <- ncol(R)
  W <- matrix(NA_real_, nrow = n, ncol = p)
  colnames(W) <- colnames(R)
  for (t in win:(n - 1)) {
    chunk <- as.matrix(R[(t - win + 1):t, , drop = FALSE])
    S <- cov(chunk)
    ev <- tryCatch(eigen(S, symmetric = TRUE, only.values = TRUE)$values, error = function(e) NA_real_)
    if (any(is.na(ev)) || length(ev) == 0 || min(ev, na.rm = TRUE) <= 1e-10) {
      w <- rep(1 / p, p)
    } else {
      inv1 <- solve(S, rep(1, p))
      w <- inv1 / sum(inv1)
    }
    W[t + 1, ] <- w
  }
  xts(W, order.by = index(R))
}

daily_rebalance_returns <- function(R, weights_xts, lag_weights = 1L) {
  ## weights_xts[t] 表示在 t 日开盘使用的权重（由 t-1 及以前信息估计）；收益用 t 到 t+1
  R <- as.xts(R)
  W <- as.xts(weights_xts)
  idx <- index(R)
  r_p <- rep(NA_real_, nrow(R))
  for (t in 1:(nrow(R) - 1)) {
    w <- as.numeric(W[t + lag_weights, ])
    if (any(is.na(w))) next
    r_p[t + 1] <- sum(w * as.numeric(R[t + 1, ]))
  }
  xts(r_p, order.by = idx)
}

w_1n <- xts(matrix(1 / ncol(R_assets), nrow = nrow(R_assets), ncol = ncol(R_assets), byrow = TRUE),
            order.by = index(R_assets))
colnames(w_1n) <- colnames(R_assets)

w_mvp <- rolling_mvp_weights(R_assets, win = WINDOW_MVP)

ret_1n <- daily_rebalance_returns(R_assets, w_1n, lag_weights = 0L)
ret_mvp <- daily_rebalance_returns(R_assets, w_mvp, lag_weights = 0L)

portfolio_metrics <- function(r_xts, rf = RF_DAILY) {
  r <- as.numeric(na.omit(r_xts))
  r_ex <- r - rf
  mu <- mean(r, na.rm = TRUE) * 252
  vol <- sd(r, na.rm = TRUE) * sqrt(252)
  sharpe <- mean(r_ex, na.rm = TRUE) / sd(r, na.rm = TRUE) * sqrt(252)
  c(mean_ann = mu, vol_ann = vol, Sharpe = sharpe)
}

bench_var <- var(as.numeric(na.omit(ret_1n)), na.rm = TRUE)

hedge_effectiveness <- function(r_xts, bench_var) {
  v <- var(as.numeric(na.omit(r_xts)), na.rm = TRUE)
  1 - v / bench_var
}

m_full_1n <- portfolio_metrics(ret_1n)
m_full_mvp <- portfolio_metrics(ret_mvp)
he_mvp <- hedge_effectiveness(ret_mvp, bench_var)

cat("全样本 1N：
"); print(round(m_full_1n, 4))
cat("全样本 MVP（滚动协方差）：
"); print(round(m_full_mvp, 4))
cat("相对 1N 的 MVP 对冲有效性（占位定义）：", round(he_mvp, 4), "
")

## without gold：去掉 GOLD 列再算一版 1N（示例）
R_no_gold <- R_assets[, colnames(R_assets) != "GOLD", drop = FALSE]
w_1n_ng <- xts(matrix(1 / ncol(R_no_gold), nrow = nrow(R_no_gold), ncol = ncol(R_no_gold), byrow = TRUE),
               order.by = index(R_no_gold))
ret_1n_ng <- daily_rebalance_returns(R_no_gold, w_1n_ng, lag_weights = 0L)
cat("全样本 1N（不含 GOLD）：
"); print(round(portfolio_metrics(ret_1n_ng), 4))


In [ ]:

## GPR 子样本内重复评价（占位面板）
subset_by_idx <- function(x, idx) {
  x <- as.xts(x)
  x[idx[index(x)], , drop = FALSE]
}

eval_block <- function(Rsub, label) {
  w1 <- xts(matrix(1 / ncol(Rsub), nrow = nrow(Rsub), ncol = ncol(Rsub), byrow = TRUE), order.by = index(Rsub))
  wm <- rolling_mvp_weights(Rsub, win = min(WINDOW_MVP, max(30L, nrow(Rsub) - 5L)))
  r1 <- daily_rebalance_returns(Rsub, w1, lag_weights = 0L)
  rm <- daily_rebalance_returns(Rsub, wm, lag_weights = 0L)
  bvar <- var(as.numeric(na.omit(r1)), na.rm = TRUE)
  m1 <- portfolio_metrics(r1)
  mm <- portfolio_metrics(rm)
  out1 <- data.frame(
    segment = label, strategy = "1N",
    mean_ann = unname(m1["mean_ann"]), vol_ann = unname(m1["vol_ann"]), Sharpe = unname(m1["Sharpe"]),
    HE_vs_1N = NA_real_
  )
  out2 <- data.frame(
    segment = label, strategy = "MVP",
    mean_ann = unname(mm["mean_ann"]), vol_ann = unname(mm["vol_ann"]), Sharpe = unname(mm["Sharpe"]),
    HE_vs_1N = hedge_effectiveness(rm, bvar)
  )
  rbind(out1, out2)
}

tab_high <- eval_block(subset_by_idx(R_assets, idx_high), "high_GPR")
tab_low  <- eval_block(subset_by_idx(R_assets, idx_low), "low_GPR")
print(round(rbind(tab_high, tab_low), 4))


## 8. 结果解读与讨论清单（Markdown）

在正式结果就绪后，按下列清单写**分析段落**（每点对应一节讨论）：

1. **连通性**：全样本与 **high-GPR** 下 **TCI** 是否上升？**黄金 NET** 的符号与幅度是否支持“风险吸收国”叙事？
2. **方向**：**FROM/TO gold** 在事件窗是否呈现**非对称**（例如对股指溢出上升、对债券溢出下降）？
3. **组合**：**MVP vs 1N** 的波动下降是否以收益为代价？**纳入黄金**是否提高 **hedging effectiveness**？
4. **地缘状态**：**high vs low GPR** 下夏普与波动排序是否**翻转**？是否与 GPR 的**阈值稳健性**（0.65–0.75）一致？
5. **局限**：占位 GPR 与随机收益**不代表**任何实证结论；替换数据后需重新报告**稳健性**与**诊断检验**。

---

### 下一步（Coding 交接）

- 将 `returns_daily_xts` 的生成替换为你的清洗脚本输出对象（保持列名为 `ASSET_LABELS`）。
- 将 `gpr_monthly_xts` 替换为 Caldara–Iacoviello 等官方 GPR 序列。
- 在 `estimate_connectedness_tvp_var()` 内接入你的 TVP–VAR–DY 代码并返回作图所需 `xts`。


In [ ]:

sessionInfo()
